# MA3627 — Workshop 5: $k$-Nearest Neighbours and Linear Regression

This workshop accompanies the Week 5 lecture. Parts A–C build the empirical risk
minimisation framework and $k$-NN for classification. Parts D–G implement OLS,
polynomial regression, and Ridge/Lasso regularisation. Part F is the bias–variance
demonstration for this workshop — watch how training and test error diverge as
model complexity increases, the same pattern that appeared for $k$ in Part B.

Take-home exercises are at the end.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(0)

---
## Part A — Loss functions and empirical risk

The lecture defines the empirical risk of a hypothesis $h$ on a sample
$\{(x_i, y_i)\}_{i=1}^n$ as
$$
\hat{R}(h) = \frac{1}{n}\sum_{i=1}^n \ell(h(x_i), y_i).
$$
We begin by implementing the two most common regression loss functions by hand, to see
concretely which constant predictor each one favours.

In [ ]:
def loss_squared(y_hat, y):
    return (y_hat - y) ** 2

def loss_absolute(y_hat, y):
    return np.abs(y_hat - y)

def empirical_risk(h_vals, y, loss_fn):
    return loss_fn(h_vals, y).mean()

y_demo = np.array([1.0, 2.5, 3.0, 4.2, 5.1])
c_vals = np.linspace(0, 6, 300)

risk_sq  = [empirical_risk(np.full_like(y_demo, c), y_demo, loss_squared)  for c in c_vals]
risk_abs = [empirical_risk(np.full_like(y_demo, c), y_demo, loss_absolute) for c in c_vals]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, risk, label, colour in zip(
        axes, [risk_sq, risk_abs], ["Squared loss", "Absolute loss"],
        ["steelblue", "darkorange"]):
    ax.plot(c_vals, risk, color=colour)
    ax.axvline(np.mean(y_demo),  color="steelblue",  linestyle="--", label=f"mean  = {np.mean(y_demo):.2f}")
    ax.axvline(np.median(y_demo), color="darkorange", linestyle=":",  label=f"median = {np.median(y_demo):.2f}")
    ax.set_xlabel("Constant predictor $c$")
    ax.set_ylabel("Empirical risk")
    ax.set_title(label)
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Mean of y:   {np.mean(y_demo):.4f}  (minimises squared loss)")
print(f"Median of y: {np.median(y_demo):.4f}  (minimises absolute loss)")

**In-class exercise.** Verify analytically that the squared-loss risk
$\frac{1}{n}\sum_i (c - y_i)^2$ is minimised by $c = \bar{y}$, by differentiating with
respect to $c$ and setting the derivative to zero.

---
## Part B — $k$-NN classification and decision boundaries

We use the Iris dataset (4 features, 3 classes, 150 samples). For visualisation we work
with the first two features only; the full four-feature model is used for the error curves.

In [ ]:
iris = load_iris(as_frame=True)
X_ir, y_ir = iris.data.values, iris.target.values

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_ir, y_ir, test_size=0.3, random_state=0, stratify=y_ir
)

sc_ir = StandardScaler()
X_tr = sc_ir.fit_transform(X_tr_raw)
X_te = sc_ir.transform(X_te_raw)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
h = 0.02
X_both = np.vstack([X_tr, X_te])
x1_min, x1_max = X_both[:, 0].min() - 0.5, X_both[:, 0].max() + 0.5
x2_min, x2_max = X_both[:, 1].min() - 0.5, X_both[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x1_min, x1_max, h), np.arange(x2_min, x2_max, h))
colours = ["#a8d8ea", "#f9c784", "#c8e6c9"]
cmap_bg = plt.matplotlib.colors.ListedColormap(colours)

for ax, k in zip(axes, [1, 5, 15, 50]):
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_tr[:, :2], y_tr)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=cmap_bg, alpha=0.7)
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="Set1", s=18, edgecolors="k", linewidths=0.4)
    ax.set_title(f"$k = {k}$", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Decision boundaries (features 1 and 2 only)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
k_vals = list(range(1, 51))
train_err, test_err = [], []
for k in k_vals:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_tr, y_tr)
    train_err.append(1 - accuracy_score(y_tr, clf.predict(X_tr)))
    test_err.append(1 - accuracy_score(y_te, clf.predict(X_te)))

plt.figure(figsize=(8, 4))
plt.plot(k_vals, train_err, label="Training error", color="steelblue")
plt.plot(k_vals, test_err,  label="Test error",     color="darkorange")
plt.xlabel("$k$")
plt.ylabel("Misclassification rate")
plt.title("$k$-NN error curves — Iris (4 features, standardised)")
plt.legend()
plt.tight_layout()
plt.show()

**In-class exercise.** The decision boundaries above become smoother as $k$ increases.
Explain why $k = 1$ always achieves zero training error. At what point on the error curve
does the bias–variance trade-off appear to be best balanced?

---
## Part C — Choosing $k$ by cross-validation

We implement 5-fold CV by hand on the Iris data so the mechanics are visible, then apply
the one-standard-error rule. This exact procedure — cross-validate over a hyperparameter
grid, then apply the 1-SE rule — is reused unchanged for $\lambda$ in Part G.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)
k_vals_cv = list(range(1, 31, 2))

cv_errors = np.zeros((len(k_vals_cv), 5))
for j, k in enumerate(k_vals_cv):
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_tr)):
        clf = KNeighborsClassifier(n_neighbors=k)
        clf.fit(X_tr[tr_idx], y_tr[tr_idx])
        cv_errors[j, fold] = 1 - accuracy_score(y_tr[va_idx], clf.predict(X_tr[va_idx]))

mean_cv = cv_errors.mean(axis=1)
se_cv   = cv_errors.std(axis=1) / np.sqrt(5)

best_idx  = mean_cv.argmin()
threshold = mean_cv[best_idx] + se_cv[best_idx]
ose_idx   = np.where(mean_cv <= threshold)[0][-1]

plt.figure(figsize=(9, 4))
plt.plot(k_vals_cv, mean_cv, "o-", color="steelblue", label="CV error")
plt.fill_between(k_vals_cv, mean_cv - se_cv, mean_cv + se_cv, alpha=0.2, color="steelblue")
plt.axhline(threshold, color="darkorange", linestyle="--", label="1-SE threshold")
plt.axvline(k_vals_cv[ose_idx], color="green", linestyle=":", label=f"1-SE choice: $k={k_vals_cv[ose_idx]}$")
plt.xlabel("$k$"); plt.ylabel("CV misclassification rate")
plt.title("5-fold CV with one-standard-error rule — Iris")
plt.legend(); plt.tight_layout(); plt.show()

print(f"CV minimiser:        k = {k_vals_cv[best_idx]:2d}  (error = {mean_cv[best_idx]:.4f})")
print(f"1-SE selected k:     k = {k_vals_cv[ose_idx]:2d}  (error = {mean_cv[ose_idx]:.4f})")

In [ ]:
param_grid = {"n_neighbors": k_vals_cv}
gs = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring="accuracy")
gs.fit(X_tr, y_tr)
k_star = gs.best_params_["n_neighbors"]

clf_final = KNeighborsClassifier(n_neighbors=k_vals_cv[ose_idx])
clf_final.fit(X_tr, y_tr)
print(f"GridSearchCV exact minimiser: k = {k_star}")
print(f"Test error (1-SE model):      {1 - accuracy_score(y_te, clf_final.predict(X_te)):.4f}")
print("The manual implementation above and GridSearchCV should agree on the minimiser.")

**In-class exercise.** The 1-SE rule selects a larger $k$ than the exact CV minimiser.
Explain in one sentence why this is generally desirable from a bias–variance perspective.

---
## Part D — OLS from scratch: the normal equations

We implement OLS directly from $\hat{\mathbf{w}} = (X^\top X)^{-1} X^\top \mathbf{y}$ and
verify against the worked example from the lecture.

In [ ]:
def add_intercept(X):
    n = X.shape[0]
    return np.hstack([np.ones((n, 1)), X])

def ols_fit(X, y):
    return np.linalg.solve(X.T @ X, X.T @ y)

def ols_predict(X, w):
    return X @ w

x = np.array([0., 1., 2., 3.])
y = np.array([1., 3., 2., 5.])
X = add_intercept(x.reshape(-1, 1))

XtX = X.T @ X
Xty = X.T @ y
w_hat = ols_fit(X, y)
print(f"OLS solution: w0 = {w_hat[0]:.4f}, w1 = {w_hat[1]:.4f}")
print(f"Fitted line:  f(x) = {w_hat[0]:.4f} + {w_hat[1]:.4f} * x")

y_hat = ols_predict(X, w_hat)
residuals = y - y_hat
print()
print("Residuals:    ", np.round(residuals, 4))
print(f"Sum of residuals:          {residuals.sum():.10f}  (should be ~0)")
print(f"Residuals . x values:      {(x * residuals).sum():.10f}  (should be ~0)")

In [ ]:
H = X @ np.linalg.inv(X.T @ X) @ X.T

print(f"Max entry of |H^2 - H|: {np.abs(H @ H - H).max():.2e}  (should be ~0, idempotent)")
print(f"Max entry of |H - H'|:  {np.abs(H - H.T).max():.2e}  (should be ~0, symmetric)")

y_hat_proj = H @ y
print()
print("y_hat via H @ y:", np.round(y_hat_proj, 4))
print("y_hat via X @ w:", np.round(y_hat, 4))
print("(Should be identical — H is the orthogonal projection onto col(X))")

---
## Part E — OLS on real data: California Housing

We apply OLS to predict median house value from eight features, and check the residual
diagnostics. This dataset and standardised split are reused in Parts F–G.

In [ ]:
feature_names = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude"
]

try:
    raw = fetch_california_housing(as_frame=True)
    df_housing = raw.frame.copy()
    X_raw = df_housing[feature_names].values
    y_raw = df_housing["MedHouseVal"].values
except Exception as e:
    print(f"California Housing fetch unavailable ({type(e).__name__}); using synthetic fallback.")
    n_synth = 3000
    rng_synth = np.random.default_rng(seed=42)
    med_inc    = rng_synth.gamma(shape=5.0, scale=0.8, size=n_synth)
    house_age  = rng_synth.uniform(1, 52, size=n_synth)
    ave_rooms  = rng_synth.normal(5.5, 1.2, size=n_synth).clip(min=1)
    ave_bedrms = ave_rooms * rng_synth.uniform(0.15, 0.35, size=n_synth)
    population = rng_synth.gamma(shape=3.0, scale=400, size=n_synth)
    ave_occup  = rng_synth.normal(3.0, 0.7, size=n_synth).clip(min=1)
    latitude   = rng_synth.uniform(32.5, 42.0, size=n_synth)
    longitude  = rng_synth.uniform(-124.3, -114.3, size=n_synth)
    med_house_val = (
        0.5 * med_inc + 0.01 * (52 - house_age) - 0.05 * ave_occup
        + rng_synth.normal(0, 0.4, size=n_synth)
    ).clip(min=0.15, max=5.0)
    X_raw = np.column_stack([
        med_inc, house_age, ave_rooms, ave_bedrms,
        population, ave_occup, latitude, longitude
    ])
    y_raw = med_house_val

print(f"Dataset shape: {X_raw.shape}")

X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_raw, y_raw, test_size=0.2, random_state=0)

sc_h = StandardScaler()
X_tr_h_sc = sc_h.fit_transform(X_tr_h)
X_te_h_sc = sc_h.transform(X_te_h)

print(f"Training: {X_tr_h_sc.shape[0]} samples  |  Test: {X_te_h_sc.shape[0]} samples")

In [ ]:
def regression_metrics(y_true, y_pred, label=""):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{label}")
    print(f"  RMSE: {rmse:.4f}   MAE: {mae:.4f}   R²: {r2:.4f}")
    return {"RMSE": rmse, "MAE": mae, "R2": r2}

lr = LinearRegression()
lr.fit(X_tr_h_sc, y_tr_h)

regression_metrics(y_tr_h, lr.predict(X_tr_h_sc), "OLS — training set")
regression_metrics(y_te_h, lr.predict(X_te_h_sc), "OLS — test set    ")

print()
print("Coefficients (standardised features):")
for name, coef in zip(feature_names, lr.coef_):
    print(f"  {name:<20} {coef:+.4f}")

In [ ]:
residuals_te = y_te_h - lr.predict(X_te_h_sc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(lr.predict(X_te_h_sc), residuals_te, s=5, alpha=0.3, color="steelblue")
axes[0].axhline(0, color="firebrick", lw=1)
axes[0].set_xlabel("Fitted values"); axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs fitted (test set)")

axes[1].hist(residuals_te, bins=60, color="steelblue", edgecolor="white", lw=0.3)
axes[1].axvline(0, color="firebrick", lw=1)
axes[1].set_xlabel("Residual"); axes[1].set_ylabel("Count")
axes[1].set_title("Residual distribution (test set)")

plt.tight_layout()
plt.show()

A well-specified linear model should produce residuals that are approximately symmetric
around zero with no pattern against fitted values. Systematic curvature or fanning
(heteroscedasticity) in the left plot signals model misspecification.

---
## Part F — Polynomial regression: the bias–variance trade-off, revisited

This is the bias–variance demonstration for this workshop. In Part B, complexity was
controlled by $k$; here it is controlled by polynomial degree $d$, and the same U-shaped
test-error pattern appears.

In [ ]:
rng_poly = np.random.default_rng(3)
n_tr = 20
n_te = 200

x_cos_tr = np.sort(rng_poly.uniform(0, 2, n_tr))
y_cos_tr = np.cos(np.pi * x_cos_tr / 2) + rng_poly.normal(0, 0.15, n_tr)

x_cos_te = np.linspace(0, 2, n_te)
y_cos_te = np.cos(np.pi * x_cos_te / 2)

degrees = [1, 2, 3, 5, 9, 14]
train_rmse = []
test_rmse  = []

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for i, d in enumerate(degrees):
    pipe = Pipeline([
        ("poly",  PolynomialFeatures(degree=d, include_bias=False)),
        ("scale", StandardScaler()),
        ("ols",   LinearRegression())
    ])
    pipe.fit(x_cos_tr.reshape(-1, 1), y_cos_tr)

    y_tr_pred = pipe.predict(x_cos_tr.reshape(-1, 1))
    y_te_pred = pipe.predict(x_cos_te.reshape(-1, 1))

    tr_rmse = np.sqrt(mean_squared_error(y_cos_tr, y_tr_pred))
    te_rmse = np.sqrt(mean_squared_error(y_cos_te, y_te_pred))
    train_rmse.append(tr_rmse)
    test_rmse.append(te_rmse)

    ax = axes[i]
    ax.scatter(x_cos_tr, y_cos_tr, s=20, color="steelblue", zorder=3, alpha=0.8)
    ax.plot(x_cos_te, y_cos_te, color="black", lw=1.2, label="True")
    ax.plot(x_cos_te, y_te_pred, color="firebrick", lw=1.2, label=f"d={d}")
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"degree = {d}  |  train RMSE = {tr_rmse:.3f}  |  test RMSE = {te_rmse:.3f}", fontsize=9)
    ax.legend(fontsize=8)
    ax.set_xlabel("x")

plt.suptitle("Polynomial regression: underfitting to overfitting", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(degrees, train_rmse, marker="o", color="steelblue", label="Training RMSE")
ax.plot(degrees, test_rmse,  marker="o", color="firebrick",  label="Test RMSE")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("RMSE")
ax.set_title("Bias–variance trade-off: training vs test RMSE")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'Degree':>7}  {'Train RMSE':>12}  {'Test RMSE':>12}")
for d, tr, te in zip(degrees, train_rmse, test_rmse):
    print(f"{d:>7}  {tr:>12.4f}  {te:>12.4f}")

Training RMSE decreases monotonically with degree. Test RMSE follows a U-shape: it falls
as degree increases from 1 to the optimal value, then rises as the model overfits. The
degree-9 and degree-14 fits oscillate wildly (Runge phenomenon) while achieving near-zero
training error — the same qualitative picture as $k=1$ in Part B, at the opposite end of
the complexity axis.

---
## Part G — Regularisation: Ridge, Lasso, and cross-validation

We apply Ridge and Lasso to the California Housing data from Part E, and select
$\lambda$ by cross-validation exactly as $k$ was selected in Part C.

In [ ]:
lambdas = np.logspace(-3, 3, 100)
coef_paths = []

for lam in lambdas:
    ridge = Ridge(alpha=lam)
    ridge.fit(X_tr_h_sc, y_tr_h)
    coef_paths.append(ridge.coef_.copy())

coef_paths = np.array(coef_paths)

fig, ax = plt.subplots(figsize=(10, 5))
for j, name in enumerate(feature_names):
    ax.plot(lambdas, coef_paths[:, j], lw=1.5, label=name)

ax.set_xscale("log")
ax.set_xlabel("lambda (regularisation parameter)")
ax.set_ylabel("Coefficient value")
ax.set_title("Ridge coefficient paths (California Housing)")
ax.axvline(1.0, color="black", ls="--", lw=0.8, label="lambda = 1")
ax.legend(fontsize=8, loc="right")
plt.tight_layout()
plt.show()

All coefficients shrink towards zero as $\lambda$ increases; features with initially
large coefficients shrink faster. This is Ridge's *shrinkage without selection* —
contrast with Lasso below.

In [ ]:
alphas_grid = np.logspace(-3, 3, 200)
ridge_cv = RidgeCV(alphas=alphas_grid, scoring="neg_root_mean_squared_error", cv=10)
ridge_cv.fit(X_tr_h_sc, y_tr_h)

print(f"Best lambda (RidgeCV): {ridge_cv.alpha_:.4f}")

ridge_best = Ridge(alpha=ridge_cv.alpha_)
ridge_best.fit(X_tr_h_sc, y_tr_h)
regression_metrics(y_te_h, ridge_best.predict(X_te_h_sc), f"Ridge (lambda={ridge_cv.alpha_:.4f}) — test set")
regression_metrics(y_te_h, lr.predict(X_te_h_sc), "OLS (no regularisation)   — test set")

In [ ]:
lasso_cv = LassoCV(cv=10, random_state=0, max_iter=5000, n_alphas=200)
lasso_cv.fit(X_tr_h_sc, y_tr_h)

print(f"Best lambda (LassoCV): {lasso_cv.alpha_:.6f}")
print()

lasso_best = Lasso(alpha=lasso_cv.alpha_, max_iter=5000)
lasso_best.fit(X_tr_h_sc, y_tr_h)

print("Lasso coefficients at selected lambda:")
for name, coef in zip(feature_names, lasso_best.coef_):
    status = "  (zeroed out)" if coef == 0 else ""
    print(f"  {name:<20} {coef:+.4f}{status}")

print()
regression_metrics(y_te_h, lasso_best.predict(X_te_h_sc), "Lasso (CV lambda) — test set")

In [ ]:
models = {"OLS": lr, "Ridge": ridge_best, "Lasso": lasso_best}

print(f"{'Model':<10}  {'RMSE':>8}  {'MAE':>8}  {'R²':>8}")
print("-" * 40)
for name, model in models.items():
    y_pred = model.predict(X_te_h_sc)
    rmse = np.sqrt(mean_squared_error(y_te_h, y_pred))
    mae  = mean_absolute_error(y_te_h, y_pred)
    r2   = r2_score(y_te_h, y_pred)
    print(f"{name:<10}  {rmse:>8.4f}  {mae:>8.4f}  {r2:>8.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
x_pos = np.arange(len(feature_names))
width = 0.25

ax.bar(x_pos - width, lr.coef_,         width, label="OLS",   color="steelblue",  alpha=0.8)
ax.bar(x_pos,         ridge_best.coef_, width, label="Ridge", color="firebrick",  alpha=0.8)
ax.bar(x_pos + width, lasso_best.coef_, width, label="Lasso", color="darkorange", alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Coefficient value")
ax.set_title("Coefficient comparison: OLS vs Ridge vs Lasso (standardised features)")
ax.axhline(0, color="black", lw=0.8)
ax.legend()
plt.tight_layout()
plt.show()

Lasso zeroes out the least useful features entirely, while Ridge shrinks every
coefficient but keeps them all nonzero — the shrinkage-vs-selection distinction from the
lecture, visible directly in the bars above.

---
## Take-home exercises

**Exercise 1.** The Wine dataset (`sklearn.datasets.load_wine`) has 13 features and 3
classes. Apply the full pipeline from Parts B–C: standardise the features, split into
training and test sets, run 5-fold cross-validation over $k \in \{1, 3, \ldots, 29\}$,
apply the one-standard-error rule to select $k$, and report the test error of the final
model. Compare your selected $k$ to the exact cross-validation minimiser.

**Exercise 2.** Implement Ridge regression from scratch using only NumPy: given
standardised design matrix $X$, target vector $\mathbf{y}$, and regularisation parameter
$\lambda$, compute $\hat{\mathbf{w}}_\lambda = (X^\top X + n\lambda I)^{-1} X^\top
\mathbf{y}$ directly. Verify your implementation matches `sklearn.linear_model.Ridge` for
several values of $\lambda$ on the California Housing data from Part E. Note the factor
of $n$ in the penalty: sklearn's `Ridge(alpha=a)` minimises $\|\mathbf{y} -
X\mathbf{w}\|^2 + a\|\mathbf{w}\|^2$ (without the $1/n$ factor), so you will need to
account for this when comparing.

**Exercise 3.** Using the synthetic cosine data from Part F, repeat the bootstrap
bias–variance experiment from Part B (empirical bias$^2$ and variance across bootstrap
resamples) but now for Ridge-regularised degree-9 polynomials. Fix $B = 200$ bootstrap
resamples and evaluate at three query points $x \in \{0.5, 1.0, 1.5\}$. Plot estimated
bias$^2$ and variance as functions of $\lambda$ on a log scale, and identify the
$\lambda$ that minimises the total error.

**Exercise 4.** The Digits dataset (`sklearn.datasets.load_digits`) has 64
pixel-intensity features. Compare $k$-NN test accuracy with and without standardisation
(features range 0–16, but background pixels are almost always zero while central pixels
reach the maximum). Then compare the Euclidean, Manhattan, and Chebyshev metrics on the
standardised data. Explain why the differences between metrics are typically small once
the data are standardised.